In [1]:
import sys
import os
notebook_dir = os.path.dirname(os.path.abspath(''))
sys.path.insert(0, os.path.abspath(os.path.join(notebook_dir, '..', 'lime_ndt')))
sys.path.insert(0, os.path.abspath(os.path.join(notebook_dir, '..')))

## Diabetes Dataset

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from lime_ndt.lime_tabular import LimeTabularExplainer as LimeNDTExplainer
from lime_ndt.utils.ndt_sklearn_wrapper import NDTRegressorWrapper
from joblib import Parallel, delayed

# ========================
# 1. Charger dataset
# ========================
data = load_diabetes()
X = data.data
y = data.target
feature_names = data.feature_names

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

# ========================
# 2. Modèle global (Random Forest ici)
# ========================
rf = RandomForestRegressor(random_state=42)
rf.fit(X_train, y_train)

def predict_fn(X):
    return rf.predict(X)

# ========================
# 3. Explainer NDT
# ========================
explainer_ndt = LimeNDTExplainer(
    X_train,
    feature_names=feature_names,
    discretize_continuous=False,
    mode='regression',
)

# ========================
# 4. Fonction de stabilité
# ========================
def explanation_stability_ndt(g1, g2, n_repeats=1, instance_id=0, num_features=None):
    explanations = []
    if num_features is None:
        num_features = X_train.shape[1]

    for seed in range(n_repeats):
        np.random.seed(seed)

        exp = explainer_ndt.explain_instance(
            X_test[instance_id],
            predict_fn,
            num_features=num_features,
            model_regressor=NDTRegressorWrapper(D=X_train.shape[1], gammas=[g1, g2], random_state=seed)
        )

        weights = dict(exp.as_list())
        vec = np.array([weights.get(f, 0) for f in feature_names])
        explanations.append(vec)

    # Moyenne de la similarité cosinus entre explications
    sims = []
    for i in range(len(explanations)):
        for j in range(i+1, len(explanations)):
            num = np.dot(explanations[i], explanations[j])
            denom = np.linalg.norm(explanations[i]) * np.linalg.norm(explanations[j])
            sims.append(num/denom if denom > 0 else 0)

    return np.mean(sims)

# ========================
# 5. Grid des gammas
# ========================
gamma1_values = np.logspace(-1, 2, 20)   # de 0.1 à 100
gamma2_values = np.logspace(-1, 2, 20)

# Choisir une instance
instance_id = 0

# ========================
# 6. Calcul parallèle
# ========================
scores = Parallel(n_jobs=-1)(
    delayed(explanation_stability_ndt)(g1, g2, n_repeats=5, instance_id=instance_id)
    for g2 in gamma2_values
    for g1 in gamma1_values
)

scores = np.array(scores).reshape(len(gamma2_values), len(gamma1_values))

# ========================
# 7. Plot du landscape
# ========================
plt.figure(figsize=(9,7))
cs = plt.contourf(
    gamma1_values,
    gamma2_values,
    scores,
    levels=50,
    cmap="viridis"
)
plt.colorbar(cs, label="Stability (cosine similarity)")
plt.xscale("log")
plt.yscale("log")
plt.xlabel("Gamma1")
plt.ylabel("Gamma2")
plt.title("Landscape of NDT explanation stability on the Diabetes dataset")
plt.show()


c:\Users\DELL\Desktop\lime_ndt\.venv\lib\site-packages\joblib\externals\loky\process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


KeyboardInterrupt: 